**Running this on Google Colab.** The first code cell fetches this entry's files and installs the packages it was run with, pinned for Colab's Python version. The results in the registry come from the entry's own environment ([requirements.lock.txt](requirements.lock.txt), Python 3.13); a different Python or platform can change the last digits of some numbers.

In [ ]:
# On Google Colab: fetch this entry's files and install its pinned packages. Elsewhere this cell does nothing.
import sys
if "google.colab" in sys.modules:
    import os, subprocess
    if not os.path.isdir("/content/encore"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", "colab", "--filter=blob:none", "--sparse",
                        "https://github.com/encorescience/encore-registry", "/content/encore"], check=True)
        subprocess.run(["git", "-C", "/content/encore", "sparse-checkout", "set", "entries/longcore-2024"], check=True)
    os.chdir("/content/encore/entries/longcore-2024")
    sys.path.insert(0, os.getcwd())
    version = f"{sys.version_info.major}.{sys.version_info.minor}"
    lock = f"requirements.colab-{version}.txt"
    if not os.path.exists(lock):
        print(f"No lock for Python {version}: installing the entry's unpinned requirements (requirements.in).")
        lock = "requirements.in"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
    subprocess.run(["uv", "pip", "install", "--system", "-q", "-r", lock], check=True)
    print("Installed. If Colab asks to restart the session, restart it and run the notebook from the top again.")


# Reproducing Longcore et al. (2024) from Dryad dataset t1g1jwtb0

**Paper.** Longcore T, Villanueva SAMB, Nguyen-Ngo K, Ghiani CA, Harrison B, Colwell CS (2024).
Relative importance of intensity and spectrum of artificial light at night in disrupting behavior
of a nocturnal rodent. *J. Exp. Biol.* 227: jeb247235.
[doi:10.1242/jeb.247235](https://doi.org/10.1242/jeb.247235). The claims file
(`claims/longcore-2024.yaml`) was written from the version of record as served by Europe PMC
(PMC11418196), main text, Methods, legends and Discussion; the cells of Table 1 were not in that
text.

**Dataset.** [DRYAD:t1g1jwtb0](https://doi.org/10.5061/dryad.t1g1jwtb0), version id 300765
(Dryad's version number 2, published 2024-06-21). The paper cites the dataset by its DOI, which
resolves to the latest version; Dryad's API lists one published version, 300765, so it is the one
pinned. The files and their SHA-256, from
[Dryad's file list](https://datadryad.org/api/v2/versions/300765/files):

| File | SHA-256 |
|---|---|
| `SuppressionDataset.csv` | 11d5cd29d536b4d9782d47a9af7ba6849434820fb923a897f94543585fc116de |
| `cFosdata.csv` | 9a77244c5b27bb0747831c5d9b6d5b5129c2ebba8528fada4a1e9311fa8dc964 |
| `SPD.csv` | 98d9ea6de15fb9bc42060c24a80ace97e5c265b1a54143b3a084c638f019c9ae |
| `README.md` | b1466b96c9e140720496e7b26ba5a79e6a41d8693a5dee4a79dbc11b623ad8be |

Dryad needs a token to download, so the workflow stages the files before the notebook runs, and
they are read in place into memory (`alan.py`); the first cell checks their SHA-256. No data come
from outside the dataset. Nothing here draws random numbers, so no seeds are used.

**Claims.** All thirteen claims are tested.

| Claim | Tested |
|---|---|
| table1-suppression-0p5-and-above | tested |
| table1-increase-0p01 | tested |
| tukey-illuminance-within-cct | tested |
| cct-0p01-not-significant | tested |
| glm-illuminance-cct-interaction | tested |
| effect-size-illuminance-over-cct | tested |
| cct-effect-grows-with-intensity | tested |
| lower-cct-less-suppression | tested |
| aicc-individual-sex | tested |
| sex-males-less-suppressed | tested |
| sex-predicted-suppression | tested |
| individual-variation | tested |
| cfos-no-cct-difference | tested |

The GLM readings the claims file names (normal distribution with a log link as the Methods state,
or an identity link; illuminance in lux or log10 lux; likelihood-ratio or Wald tests) are all run.

In [ ]:
import time
import warnings

import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.stats.multicomp import pairwise_tukeyhsd

import alan as a

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
started = time.time()
print("Dryad dataset", a.DRYAD_ID, "version", a.DRYAD_VERSION)
for name, digest in a.check_staged().items():
    print(f"{name}: SHA-256 {digest} (matches Dryad's file list)")
print()
print(a.read_bytes("README.md").decode("utf-8"))

## Data check

Each check of the description against the values is stated under its output. The dataset has no
time series: each table holds one value per mouse and treatment, or per animal, so there is no
clock to align with a trial table.

In [ ]:
raw = a.read_csv("SuppressionDataset.csv")
print("SuppressionDataset.csv:", raw.shape, "columns", list(raw.columns))
print(raw.dtypes.to_string())
print("missing values:", raw.isna().sum().to_dict())
d = a.suppression()
print("\nIlluminance values (lx):", sorted(d.lux.unique()), " CCT values (K):", sorted(d.cct.unique()))
print("Sex values:", d.sex.value_counts().to_dict())
print("Mice:", d.mouse.nunique(), "; sex per mouse constant:", bool(d.groupby("mouse").sex.nunique().eq(1).all()))
print("Mice by sex:", d.groupby("sex").mouse.nunique().to_dict())
print("\nSuppression: min", d.supp.min(), " max", d.supp.max(), " mean", round(d.supp.mean(), 2),
      "; values above 100:", int((d.supp > 100).sum()), "; values exactly 100:", int((d.supp == 100).sum()))
print("\nRows per treatment (mice):")
print(d.pivot_table(index="lux", columns="cct", values="supp", aggfunc="count").to_string())
print("\nRows per mouse:")
print(d.groupby(["mouse", "sex"]).size().to_string())
print("\nRows repeated for the same mouse and treatment:", int(d.duplicated(["lux", "cct", "mouse"]).sum()))
wide = d.pivot_table(index="mouse", columns=["lux", "cct"], values="supp")
same = [(m1, m2) for i, m1 in enumerate(wide.index) for m2 in wide.index[i + 1:]
        if np.allclose(wide.loc[m1].dropna(), wide.loc[m2].reindex(wide.loc[m1].dropna().index), equal_nan=False)]
print("Pairs of mice with identical suppression in every shared treatment:", same)
r = wide.T.corr().to_numpy(copy=True)
np.fill_diagonal(r, np.nan)
print("Largest correlation between two mice over shared treatments:", round(float(np.nanmax(r)), 3))

The illuminances (0.01, 0.5, 5, 50 lx), CCTs (1750, 1950, 3000 K), the 16 mice and the sexes
(8 female, 8 male) are those of the Methods; each mouse has one sex throughout. No mouse and
treatment is deposited twice, and no two mice share their values. Suppression runs from -59.6 to
100, so it is a percent, not the "-1 to 1" the README gives (finding below). The file holds 183 of
the 192 mouse-by-treatment values; the missing ones are shown in the next cells. The file holds
the suppression values only, not the wheel counts they were computed from, so the computation
(prior-day ZT14 to 15 against exposure-day ZT14 to 15) cannot be checked, and nothing shows
whether a mouse with no prior-day activity was left out.

In [ ]:
d = a.suppression()
print("README says Suppression 'ranges -1 to 1'; the values range from", d.supp.min(), "to", d.supp.max())
print("values with |Suppression| > 1:", int((d.supp.abs() > 1).sum()), "of", len(d))

In [ ]:
d = a.suppression()
grid = pd.MultiIndex.from_product([sorted(d.mouse.unique()), a.LUX, a.CCTS], names=["mouse", "lux", "cct"])
missing = grid.difference(d.set_index(["mouse", "lux", "cct"]).index)
print("Mouse-by-treatment values expected (16 mice x 12 treatments):", len(grid), "; deposited:", len(d),
      "; missing:", len(missing))
print(missing.to_frame(index=False).to_string(index=False))
print("\nMice per treatment (the Fig. 2A legend gives n = 16):")
print(d.pivot_table(index="lux", columns="cct", values="supp", aggfunc="count").to_string())

Nine values are missing: mouse 6FC1, P2 at all six 0.01 and 0.5 lx treatments and mouse 6FC2, P8
at the three 0.01 lx treatments. The Methods give the exposure order 5, 0.5, 0.01, 50 lx, so these
are consecutive weeks of the schedule. Neither the README nor the paper says why they are missing;
the paper gives n = 16 mice for every treatment (Fig. 2A legend).

In [ ]:
cf = a.read_csv("cFosdata.csv")
print("cFosdata.csv:", cf.shape, "columns", list(cf.columns))
print(cf.groupby("CCT")["Number cFos+ Cells"].agg(["count", "min", "max", "mean"]).round(3).to_string())
print("\nColumns for animal, sex, section, side or observer:",
      [c for c in cf.columns if c not in ("CCT", "Number cFos+ Cells")])
print("Values that are not whole numbers:", int((cf["Number cFos+ Cells"] % 1 != 0).sum()), "of", len(cf))
spd = a.read_csv("SPD.csv")
print("\nSPD.csv:", spd.shape, "columns", list(spd.columns))
print(spd.groupby("Lamp").agg(n=("Wavelength", "size"), nm_min=("Wavelength", "min"), nm_max=("Wavelength", "max"),
                              step=("Wavelength", lambda w: float(np.unique(np.diff(np.sort(w))).max())),
                              spd_min=("SPD", "min"), spd_max=("SPD", "max"),
                              peak_nm=("Wavelength", lambda w: int(w.loc[spd.loc[w.index, "SPD"].idxmax()]))).to_string())

The cFos file holds 6 values per CCT (1950 and 3000 K), as the Methods give (n = 3 per sex per
light). All twelve values are fractions, as an average over sections and sides would be:
the file holds the per-animal value the Methods describe ("one value per animal"), with no animal
identifier, sex, section, side or observer. The SPD file holds the three lamps at 1-nm steps from
360 to 780 nm, scaled to a maximum near 1, each peaking at 644 nm; no claim uses it.

## table1-suppression-0p5-and-above

> Light of 0.5 lx and brighter reduced mouse activity during the 1-h exposure at every CCT, except
> 0.5 lx at 1750 K.

Criterion: per treatment, the mean over mice of percent suppression; reproduced when it is above
zero for all eight treatments at 0.5, 5 and 50 lx other than 0.5 lx/1750 K.

In [ ]:
d = a.suppression()
cm = a.cell_means(d)
cm["mice > 0"] = d.assign(pos=d.supp > 0).groupby(["lux", "cct"]).pos.sum()
by_sex = d.pivot_table(index=["lux", "cct"], columns="sex", values="supp", aggfunc="mean").add_prefix("mean ")
table = cm.join(by_sex).round(2)
print("Percent suppression per treatment (positive: less activity), over the mice that have it:")
print(table.to_string())
abstract = {(50.0, 3000): 91, (50.0, 1750): 53, (0.01, 1750): -12.3, (0.01, 3000): -3}
print("\nThe four Table 1 values the abstract gives (as suppression), against the means here:",
      {f"{t[0]} lx {t[1]} K": (v, round(float(cm.loc[t, "mean"]), 2)) for t, v in abstract.items()})
named = [(lx, c) for lx in (0.5, 5.0, 50.0) for c in a.CCTS if (lx, c) != (0.5, 1750)]
above = {t: bool(cm.loc[t, "mean"] > 0) for t in named}
print("\nMean above zero at the eight treatments named:", sum(above.values()), "of 8")
for t, ok in above.items():
    print(f"  {t[0]:>5} lx {t[1]} K: mean {cm.loc[t, 'mean']:6.2f}, median {cm.loc[t, 'median']:6.2f}, "
          f"one-sample t-test against 0 p = {stats.ttest_1samp(d[(d.lux == t[0]) & (d.cct == t[1])].supp, 0).pvalue:.3g}")

**Result: reproduced.** All eight means are above zero. The means are over 15 mice at 0.5 lx
and 16 at 5 and 50 lx, where the criterion says 16, as one mouse's 0.5 lx values are not in the
dataset. The outcome is judged on the sign of the mean, as the criterion says, and three of the
eight cannot be told from zero: at 0.5 lx the means at 1950 K (0.79) and 3000 K (2.98) are barely
above zero, their medians are 0.67 and 1.53, and neither differs from zero by a one-sample t-test
(p 0.84 and 0.49); nor does 5 lx at 1750 K (p 0.18). At 5 and 50 lx every mean is 7.0 or more. The
abstract's values for 50 lx (91% at 3000 K, 53% at 1750 K) match the means here (90.44 and 53.73),
and so do its 0.01 lx values (printed above).


## table1-increase-0p01

> All three 0.01 lx treatments, and the 0.5 lx/1750 K treatment, increased mouse activity during
> the exposure.

Criterion: the mean over mice of percent suppression below zero for all four treatments named.

In [ ]:
d = a.suppression()
cm = a.cell_means(d)
named = [(0.01, 1750), (0.01, 1950), (0.01, 3000), (0.5, 1750)]
for t in named:
    s = d[(d.lux == t[0]) & (d.cct == t[1])].supp
    print(f"{t[0]:>5} lx {t[1]} K: n {len(s)}, mean {s.mean():6.2f}, median {s.median():6.2f}, mice below 0: "
          f"{int((s < 0).sum())}, by sex F {s[d.sex == 'F'].mean():6.2f} M {s[d.sex == 'M'].mean():6.2f}")
    print(f"    one-sample t-test against 0 p = {stats.ttest_1samp(s, 0).pvalue:.3g}")
abstract = {(0.01, 1750): -12.3, (0.01, 3000): -3}
print("\nThe abstract's values (as suppression), against the means here:",
      {f"{t[0]} lx {t[1]} K": (v, round(float(cm.loc[t, "mean"]), 2)) for t, v in abstract.items()})
print("\nMean below zero at the four treatments named:", sum(cm.loc[t, "mean"] < 0 for t in named), "of 4")

**Result: reproduced.** The four means are below zero (-12.32, -4.71, -3.04 at 0.01 lx for 1750,
1950 and 3000 K, and -2.66 at 0.5 lx/1750 K). The means are over 14, 14, 14 and 15 mice, where the
criterion says 16, as nine values are not in the dataset. The abstract's 12.3% and 3% increases at
0.01 lx match. The outcome is judged on the sign of the mean, as the criterion says, and none of the four
means differs from zero by a one-sample t-test (p 0.055 at 0.01 lx/1750 K, 0.32 to 0.67 at the
other three; printed above); at 0.01 lx/3000 K the median is 0.03 and 7 of 14 mice are below
zero. The means of the females at 0.01 lx/3000 K (0.78) and 0.5 lx/1750 K
(3.53) are above zero, so at these two treatments the increases come from the males; at 0.01 lx/1750 K
and 1950 K the female means are also below zero (-2.50 and -0.58).


## tukey-illuminance-within-cct

> In Tukey-Kramer comparisons within color temperatures, the effect of illuminance on activity was
> significantly greater only for 50 lx at 1750 K.

Criterion: Tukey-Kramer HSD on per-mouse suppression, alpha 0.05, treatments as independent groups.
Reading (a): within each CCT the four illuminances are compared; reproduced if at 1750 K 50 lx
differs from the lower illuminances and at 1950 and 3000 K no pair differs. Reading (b): within
each illuminance the three CCTs are compared; reproduced if only at 50 lx do CCTs differ, with 1750
K differing from the others. One Tukey family per CCT (a) or per illuminance (b), as "holding the
other constant" (Methods, paragraph 3) says; a single family over all twelve treatments is shown
as a variant.

In [ ]:
d = a.suppression()

def tukey(sub, by):
    r = pairwise_tukeyhsd(sub.supp, sub[by].astype(str), alpha=0.05)
    t = pd.DataFrame(r._results_table.data[1:], columns=r._results_table.data[0])
    return t

print("Reading (a): within each CCT, the four illuminances")
sig_a = {}
for c in a.CCTS:
    t = tukey(d[d.cct == c], "lux")
    sig_a[c] = t[t.reject][["group1", "group2"]].values.tolist()
    print(f"\nCCT {c} K"); print(t.to_string(index=False))
print("\nReading (b): within each illuminance, the three CCTs")
sig_b = {}
for lx in a.LUX:
    t = tukey(d[d.lux == lx], "cct")
    sig_b[lx] = t[t.reject][["group1", "group2"]].values.tolist()
    print(f"\n{lx} lx"); print(t.to_string(index=False))
print("\nSignificant pairs, reading (a):", sig_a)
print("Significant pairs, reading (b):", sig_b)
ok_a = (all(["50.0" in p for p in sig_a[1750]]) and len(sig_a[1750]) > 0 and not sig_a[1950] and not sig_a[3000])
ok_b = (not sig_b[0.01] and not sig_b[0.5] and not sig_b[5.0] and len(sig_b[50.0]) > 0
        and all("1750" in p for p in sig_b[50.0]))
print("Reading (a) matches the sentence:", ok_a, "; reading (b) matches:", ok_b)

In [ ]:
d = a.suppression()
lab = d.lux.astype(str) + " lx " + d.cct.astype(str) + " K"
r = pairwise_tukeyhsd(d.supp, lab, alpha=0.05)
t = pd.DataFrame(r._results_table.data[1:], columns=r._results_table.data[0])
same_cct = t[t.group1.str.split(" lx ").str[1] == t.group2.str.split(" lx ").str[1]]
same_lux = t[t.group1.str.split(" lx ").str[0] == t.group2.str.split(" lx ").str[0]]
print("Variant: one Tukey-Kramer family over all 12 treatments (66 pairs);", int(t.reject.sum()), "pairs differ")
print("\nPairs within a CCT:"); print(same_cct.to_string(index=False))
print("\nPairs within an illuminance:"); print(same_lux.to_string(index=False))

**Result: not reproduced.** Neither reading matches the sentence. Under reading (a), 50 lx differs
from every lower illuminance at all three CCTs (adjusted p < 0.001), and at 3000 K 5 lx also
differs from 0.01 and 0.5 lx; the comparison at 1750 K is not the only one with a difference.
Under reading (b), CCTs differ at 5 lx and at 50 lx, and in both the pairs that differ are 3000 K
against 1750 K and against 1950 K (at 50 lx mean differences 36.7 and 29.5 points, adjusted p
0.0003 and 0.0035); 1750 K and 1950 K do not differ (7.2 points, p 0.68). In the single family over
twelve treatments (variant) the within-CCT pairs that differ are the same, and within illuminances
3000 K differs from 1750 K at 5 lx (p 0.0498) and from both other CCTs at 50 lx. The data show
3000 K standing apart at the higher illuminances, not 1750 K.

## cct-0p01-not-significant

> At 0.01 lx, CCT had no significant effect on activity, although 1750 K light increased activity
> more than 3000 K light.

Criterion (a claim of absence, on the paper's test): Tukey-Kramer HSD over the three CCTs at 0.01
lx, alpha 0.05; reproduced when no pair differs and the mean increase at 1750 K is larger than at
3000 K.

In [ ]:
d = a.suppression()
s = d[d.lux == 0.01]
r = pairwise_tukeyhsd(s.supp, s.cct.astype(str), alpha=0.05)
print(r)
m = s.groupby("cct").supp.agg(["count", "mean", "std"])
print(m.round(2).to_string())
print("\nMean increase in activity (minus suppression): 1750 K", round(-m.loc[1750, "mean"], 2),
      "; 3000 K", round(-m.loc[3000, "mean"], 2), "; difference", round(m.loc[3000, "mean"] - m.loc[1750, "mean"], 2),
      "points; pairs that differ:", int(r.reject.sum()))
print("One-way ANOVA over the three CCTs at 0.01 lx: p =", round(stats.f_oneway(*[g.supp for _, g in s.groupby("cct")]).pvalue, 3))
for sex, g in s.groupby("sex"):
    print(f"  {sex}: mean suppression by CCT", g.groupby("cct").supp.mean().round(2).to_dict())

**Result: reproduced.** No CCT pair differs at 0.01 lx (smallest adjusted p 0.51; one-way ANOVA p
0.50), and the mean increase in activity is 12.32% at 1750 K against 3.04% at 3000 K, a difference
of 9.28 points.

## glm-illuminance-cct-interaction

> In a generalized linear model of suppression, illuminance, CCT and their interaction each
> contributed significantly, and the whole model was significant.

Criterion: the GLM of the claims file's opening comment on all mouse-by-treatment suppressions;
reproduced when the whole model has p < 0.001 and illuminance, CCT and the interaction have p below
0.0001, 0.0001 and 0.002 under one reading (lux or log10 lux; likelihood-ratio or Wald; log or
identity link); partially reproduced when all three terms have p < 0.05 under one reading.

The terms are coded as JMP codes them by default (effect coding, illuminance centered in the
crossing; `alan.py`). Centering changes the CCT main-effect test when the interaction is in the
model, so the uncentered fit is shown as a variant. The log link is fitted from four starting
points; all are printed.

In [ ]:
d = a.suppression()
rows, fits = [], {}
for link in ("log", "identity"):
    for x in ("lux", "loglux"):
        for center in (True, False):
            t = a.effect_tests(d, x, link, center=center)
            f = t.attrs["fit"]
            fits[(link, x, center)] = f
            for _, r in t.iterrows():
                rows.append({"link": link, "illuminance": x, "centered": center, "term": r.term, "df": r.df,
                             "LR chi2": r["LR chi2"], "LR p": r["LR p"], "Wald chi2": r["Wald chi2"], "Wald p": r["Wald p"]})
glm = pd.DataFrame(rows)
with pd.option_context("display.float_format", "{:.3g}".format):
    print(glm.to_string(index=False))
print("\nLog-link fits, SSE and success at each of the four starts:")
for k, f in fits.items():
    if f.starts:
        print(" ", k, [(round(s, 3), ok) for s, ok, _ in f.starts])

In [ ]:
bounds = {"CCT": 1e-4, "illuminance": 1e-4, "CCT x illuminance": 2e-3}
summary = []
for (link, x, center), g in glm.groupby(["link", "illuminance", "centered"], sort=False):
    g = g.set_index("term")
    for test in ("LR", "Wald"):
        p = {t: g.loc[t, f"{test} p"] for t in bounds}
        whole = g.loc["whole model", "LR p"]
        summary.append({"link": link, "illuminance": x, "centered": center, "test": test,
                        "whole p": whole, **{f"p {t}": v for t, v in p.items()},
                        "all below bounds": bool(whole < 1e-3 and all(p[t] < b for t, b in bounds.items())),
                        "all three < 0.05": bool(all(v < 0.05 for v in p.values()))})
summary = pd.DataFrame(summary)
with pd.option_context("display.float_format", "{:.3g}".format):
    print(summary.to_string(index=False))
print("\nReadings meeting every bound:", int(summary["all below bounds"].sum()),
      "; readings with all three terms p < 0.05:", int(summary["all three < 0.05"].sum()))
print("Smallest interaction p over all readings:", "{:.3g}".format(glm[glm.term == "CCT x illuminance"][["LR p", "Wald p"]].min().min()))

In [ ]:
print("Log-link fits: how well the parameters are determined")
for (link, x, center), f in fits.items():
    if link != "log" or not center:
        continue
    J = f.jacobian(f.X)
    ev = np.linalg.eigvalsh(J.T @ J)
    print(f"  {x}: smallest fitted mean {f.mu.min():.3f}, condition number of J'J {ev.max() / ev.min():.3g}")
    print("    parameters:", f.params().round(2).to_dict())
    print("    standard errors:", f.se().round(2).to_dict())
    print("    fitted mean per treatment:", d.assign(mu=f.mu).groupby(["lux", "cct"]).mu.mean().round(3).to_dict())


In [ ]:
rows = []
for extra in ("sex", "mouse"):
    for link in ("identity", "log"):
        for x in ("lux", "loglux"):
            t = a.effect_tests(d, x, link, extra=extra).set_index("term")
            for test in ("LR", "Wald"):
                p = {k: t.loc[k, f"{test} p"] for k in bounds}
                rows.append({"model": "with " + extra, "link": link, "illuminance": x, "test": test,
                             "whole p": t.loc["whole model", "LR p"], **{f"p {k}": v for k, v in p.items()},
                             "all below bounds": bool(t.loc["whole model", "LR p"] < 1e-3 and all(p[k] < b for k, b in bounds.items()))})
print("Variant, the same tests in the models with sex or mouse identity added (centered):")
with pd.option_context("display.float_format", "{:.3g}".format):
    print(pd.DataFrame(rows).to_string(index=False))


**Result: partially reproduced.** Under no reading of the claims file's model (CCT, illuminance
and their interaction) does the interaction reach the paper's bound (p < 0.002): its smallest p is
0.0063, with the identity link and illuminance in lux (likelihood ratio 0.0063, Wald 0.0065).
Under that reading the whole model (p far below 0.001), illuminance and CCT (p 2e-6 centered, 0.034
uncentered) are significant and all three terms have p < 0.05, which is the partial criterion.
With the log link the Methods state, the interaction is not significant under any reading
(smallest likelihood-ratio p 0.082). With log10 lux and the identity link the interaction p is
0.026 to 0.027.

The log-link fit with lux is not usable for Wald tests or intervals: its fitted means at 1750 K
are near zero up to 5 lx, J'J has a condition number of about 3e9 and the parameters have standard
errors of 0.6 to 43 on the log scale, so the Wald p (0.89 for illuminance against a likelihood-ratio
p of 4e-19) and the intervals built from it (the upper bound of 1e8 in sex-predicted-suppression)
say nothing about the data. The likelihood-ratio tests compare minimized sums of squares and do
not depend on those standard errors. With log10 lux the log-link fit is better determined
(condition number about 6e3).

Variant, models with sex or mouse identity. The paper's p < 0.002 may come from one of the models of
its next paragraph. In the model with mouse identity, identity link and lux, every bound is met
(interaction p 0.0007 likelihood ratio and 0.0013 Wald; CCT 1e-8, illuminance 5e-54); in the model
with sex the interaction p is 0.0038. The criterion names the model without either, so these do
not change the outcome; whether the claim's model can be the one with mouse identity is asked of a
reader in `questions/longcore-2024.yaml`.


## effect-size-illuminance-over-cct

> Over the tested ranges, the effect of illuminance (0.01 to 50 lx) on activity was several times
> greater than that of CCT (1750 to 3000 K).

Criterion: scaled estimates (each factor centered at its mean and scaled by half its range).
Reading (a), the largest absolute CCT parameter under effect coding against the scaled
illuminance estimate; reading (b), CCT refitted as a continuous term in kelvin, centered and
scaled by 625 K. Illuminance in lux or log10 lux. Reproduced when the ratio is at least 2 under one
reading. Both links are shown.

In [ ]:
d = a.suppression()
rows = []
for link in ("identity", "log"):
    for x in ("lux", "loglux"):
        for cc, reading in ((False, "a"), (True, "b")):
            X, _ = a.design(d, x, cct_continuous=cc)
            f = a.fit(d.supp, X, link)
            s = a.scaled_estimates(f, d, x, cct_continuous=cc)
            rows.append({"link": link, "illuminance": x, "reading": reading,
                         "illuminance scaled": s["illuminance"], "CCT scaled": s["CCT"],
                         "CCT levels": s.get("CCT levels"),
                         "ratio": abs(s["illuminance"]) / abs(s["CCT"])})
es = pd.DataFrame(rows)
with pd.option_context("display.float_format", "{:.3f}".format, "display.max_colwidth", 80):
    print(es.to_string(index=False))
print("\nRatios of at least 2:", int((es.ratio >= 2).sum()), "of", len(es),
      "; identity-link ratios range", round(es[es.link == "identity"].ratio.min(), 2), "to",
      round(es[es.link == "identity"].ratio.max(), 2))

**Result: reproduced.** With the identity link the illuminance effect is 3.1 to 3.8 times the CCT
effect under both readings and both scales of illuminance (illuminance 34.1 or 35.2 points against
CCT 11.0 or 9.2). With the log link the ratio is 2.9 and 3.6 with log10 lux, and 0.8 and 1.8 with
lux. The claim depends on glm-illuminance-cct-interaction, which is partially reproduced; the
effect sizes do not rest on its interaction test. The abstract's ratio of 3.6 is within the range
found.

## cct-effect-grows-with-intensity

> The influence of CCT on activity was smallest at the lowest illuminance and grew as illuminance
> increased, as shown by the significant CCT x illuminance interaction.

Criterion: the spread of mean suppression across the three CCTs (largest minus smallest
per-treatment mean) at each illuminance. Reproduced when the spread at 0.01 lx is the smallest of
the four, the spread at 50 lx is larger than at 0.01 lx, and the GLM interaction is significant.
The CCT spread predicted by the GLM at each illuminance is a variant.

In [ ]:
d = a.suppression()
cm = a.cell_means(d)["mean"].unstack("cct")
spread = cm.max(axis=1) - cm.min(axis=1)
print("Mean suppression per treatment:"); print(cm.round(2).to_string())
print("\nSpread across CCTs (largest minus smallest mean) per illuminance:"); print(spread.round(2).to_string())
print("\nSmallest spread at:", spread.idxmin(), "lx; spread at 0.01 lx smallest:", bool(spread.idxmin() == 0.01),
      "; spread at 50 lx larger than at 0.01 lx:", bool(spread[50.0] > spread[0.01]))
print("\nSpread by sex:")
for sex, g in d.groupby("sex"):
    m = g.groupby(["lux", "cct"]).supp.mean().unstack("cct")
    print(f"  {sex}:", (m.max(axis=1) - m.min(axis=1)).round(2).to_dict())
print("\nVariant, the CCT spread of the GLM's predicted means at each illuminance:")
for link in ("identity", "log"):
    for x in ("lux", "loglux"):
        X, _ = a.design(d, x)
        f = a.fit(d.supp, X, link)
        grid = pd.DataFrame([(lx, c) for lx in a.LUX for c in a.CCTS], columns=["lux", "cct"])
        grid["loglux"] = np.log10(grid.lux)
        Xg, _ = a.design(grid, x, means={"lux": d.lux.mean(), "loglux": d.loglux.mean()})
        grid["pred"] = f.predict(Xg)["mean"].values
        p = grid.pivot(index="lux", columns="cct", values="pred")
        print(f"  {link} link, {x}:", (p.max(axis=1) - p.min(axis=1)).round(2).to_dict())
p_int = glm[(glm.term == "CCT x illuminance") & (glm.link == "identity") & (glm.illuminance == "lux") & glm.centered]
print("\nInteraction p (identity link, lux, from glm-illuminance-cct-interaction): LR",
      "{:.3g}".format(p_int["LR p"].iloc[0]), ", Wald", "{:.3g}".format(p_int["Wald p"].iloc[0]))

**Result: not reproduced.** The spread across CCTs is 9.28 points at 0.01 lx, 5.64 at
0.5 lx, 23.87 at 5 lx and 36.71 at 50 lx. The smallest spread is at 0.5 lx, not at 0.01 lx, so the
first condition of the criterion fails, and the criterion gives no partial outcome. The spread at
50 lx is larger than at 0.01 lx, and the spread grows from the two lower illuminances to the two
higher ones. The interaction is significant at 0.05 (p 0.0063 with the identity link and lux) but
not at the paper's p < 0.002. In the GLM variant the predicted spread grows with illuminance and is
smallest at 0.01 lx under every reading, but a model with a term linear in illuminance for each CCT
makes the spread grow almost by construction, so the variant carries little weight. The paper's
sentence follows the GLM result and may refer to the model; whether the raw spread is the right
measure is asked of a reader in `questions/longcore-2024.yaml`. The outcome follows the criterion
as written.


## lower-cct-less-suppression

> At equal illuminance in lux, lower CCT light suppressed mouse activity less than higher CCT
> light.

Criterion: per-treatment mean suppression; reproduced when the 1750 K mean is below the 3000 K
mean at each of 0.5, 5 and 50 lx.

In [ ]:
d = a.suppression()
cm = a.cell_means(d)["mean"].unstack("cct")
out = cm.loc[[0.5, 5.0, 50.0]].assign(**{"3000 minus 1750": lambda t: t[3000] - t[1750],
                                         "1750 below 3000": lambda t: t[1750] < t[3000]})
print(out.round(2).to_string())
print("\nIlluminances (of 0.5, 5, 50) where 1750 K is below 3000 K:", int(out["1750 below 3000"].sum()), "of 3")
print("\nBy mouse: share of mice with 1750 K below 3000 K at each illuminance (mice with both):")
w = d.pivot_table(index="mouse", columns=["lux", "cct"], values="supp")
for lx in (0.5, 5.0, 50.0):
    diff = (w[(lx, 3000)] - w[(lx, 1750)]).dropna()
    print(f"  {lx} lx: {int((diff > 0).sum())} of {len(diff)}, paired Wilcoxon p = {stats.wilcoxon(diff).pvalue:.3g}")

**Result: reproduced.** The 1750 K mean is below the 3000 K mean at all three illuminances, by
5.64 points at 0.5 lx, 23.87 at 5 lx and 36.71 at 50 lx. The 1950 K means lie between them. Mouse
by mouse, 1750 K is below 3000 K in 15 of 16 mice at 5 lx and 16 of 16 at 50 lx; at 0.5 lx it is 9
of 15 and not significant (paired Wilcoxon p 0.33).

## aicc-individual-sex

> Adding sex, or mouse identity, to the GLM improved it; the model with mouse identity had the
> lowest AICc (1608.1), then the model with sex (1624.6), then the model with neither (1640.3).

Criterion: the base GLM, with sex added as a main effect, and with mouse identity added as a
categorical main effect; reproduced when the order is identity < sex < neither and the two
differences (15.7 +/- 1.6 and 16.5 +/- 1.7) are within tolerance under one reading; partially
reproduced when the order holds. AICc counts the scale as a parameter; the other count is shown.

In [ ]:
d = a.suppression()
rows = []
for link in ("log", "identity"):
    for x in ("lux", "loglux"):
        r = {"link": link, "illuminance": x}
        for extra, name in ((None, "neither"), ("sex", "sex"), ("mouse", "mouse")):
            X, _ = a.design(d, x, extra)
            f = a.fit(d.supp, X, link)
            r[name] = a.aicc(f)
            r[name + " (scale not counted)"] = a.aicc(f, count_scale=False)
            r[name + " converged"] = f.converged
        r["neither - sex"] = r["neither"] - r["sex"]
        r["sex - mouse"] = r["sex"] - r["mouse"]
        r["order holds"] = r["mouse"] < r["sex"] < r["neither"]
        r["differences in tolerance"] = abs(r["neither - sex"] - 15.7) <= 1.6 and abs(r["sex - mouse"] - 16.5) <= 1.7
        rows.append(r)
aic = pd.DataFrame(rows)
cols = ["link", "illuminance", "mouse", "sex", "neither", "neither - sex", "sex - mouse", "order holds", "differences in tolerance"]
print(aic[cols].round(2).to_string(index=False))
print("\nWith the scale not counted:")
print(aic[["link", "illuminance", "mouse (scale not counted)", "sex (scale not counted)", "neither (scale not counted)"]].round(2).to_string(index=False))
print("\nOptimizer reported success for every fit:", bool(aic.filter(like="converged").all().all()), "; n =", len(d))
print("(For the log link this does not mean the parameters are determined; see glm-illuminance-cct-interaction.)")

**Result: reproduced.** With the identity link and illuminance in lux the AICc values are 1612.3
(mouse), 1628.3 (sex) and 1643.9 (neither): the order holds and the differences are 15.6 and 16.0,
within 15.7 +/- 1.6 and 16.5 +/- 1.7. The values are 3.6 to 4.2 above the paper's (2.2 to 2.6
lower if the scale is not counted), which may reflect how JMP counts parameters. With the log link the
Methods state, the order fails with lux (mouse 1651.9 above sex 1651.3) and the differences are
6.5 and 3.3 with log10 lux; with the identity link and log10 lux the order fails.
The source of the offset of about 4 is not settled here: the paper's n of 192 cannot be
rerun, as nine values are not in the dataset, and JMP's parameter count for AICc is not stated;
both counts are shown.

## sex-males-less-suppressed

> In the model including sex, males' activity was suppressed significantly less by light at night
> than females'.

Criterion: the sex term of the GLM with sex; reproduced when p < 0.0001 with lower suppression in
males under one reading; partially reproduced when p < 0.05 with males lower.

In [ ]:
d = a.suppression()
rows = []
for link in ("identity", "log"):
    for x in ("lux", "loglux"):
        t = a.effect_tests(d, x, link, extra="sex").set_index("term")
        f = t.attrs["fit"]
        rows.append({"link": link, "illuminance": x, "sex[F] parameter": f.params()["sex[F]"],
                     "males lower": bool(f.params()["sex[F]"] > 0),
                     "LR p": t.loc["sex", "LR p"], "Wald p": t.loc["sex", "Wald p"]})
with pd.option_context("display.float_format", "{:.3g}".format):
    print(pd.DataFrame(rows).to_string(index=False))
print("\nRaw mean suppression by sex over all treatments:", d.groupby("sex").supp.mean().round(2).to_dict())
per_mouse = d.groupby(["mouse", "sex"]).supp.mean().reset_index()
print("Mean over each mouse's treatments, by sex:", per_mouse.groupby("sex").supp.agg(["count", "mean"]).round(2).to_dict())
print("Mann-Whitney on per-mouse means, 8 F against 8 M: p =",
      round(stats.mannwhitneyu(*[g.supp for _, g in per_mouse.groupby("sex")]).pvalue, 4))

**Result: reproduced.** With the identity link and lux, females are 6.3 points above the mean of
the sexes (males 12.6 points below females), likelihood-ratio p 2.4e-05 and Wald p 2.2e-05, both
below 0.0001. Males are lower under every reading; with the log link the sex term has p 0.003 to
0.007, above the paper's bound. Taken one value per mouse, the 8 males and 8 females do not differ significantly by a
Mann-Whitney test (p 0.083, printed above); the model's p treats each mouse-by-treatment value as
independent, which is the paper's method and why its p for sex is so small.

## sex-predicted-suppression

> From the model with sex, at 0.01 lx and 1750 K male activity increased by 9.3% while female
> activity was suppressed by 3.4%; at 50 lx and 3000 K male activity was suppressed by 85.7% and
> female activity by 98.3%.

Criterion: predicted suppression from the GLM with sex, for each sex at the two settings, within
the half-widths of the paper's 95% CIs (-9.3 +/- 5.45, 3.4 +/- 5.5, 85.7 +/- 10.05, 98.3 +/- 10.0),
under one model reading; partially reproduced when three are. Raw means by sex are a variant.

In [ ]:
d = a.suppression()
expected = pd.DataFrame({"lux": [0.01, 0.01, 50.0, 50.0], "cct": [1750, 1750, 3000, 3000], "sex": ["M", "F", "M", "F"],
                         "paper": [-9.3, 3.4, 85.7, 98.3], "tol": [5.45, 5.5, 10.05, 10.0]})
expected["loglux"] = np.log10(expected.lux)
rows = []
for link in ("identity", "log"):
    for x in ("lux", "loglux"):
        X, _ = a.design(d, x, "sex")
        f = a.fit(d.supp, X, link)
        Xn, _ = a.design(expected.assign(mouse="x"), x, "sex", means={"lux": d.lux.mean(), "loglux": d.loglux.mean()})
        pr = f.predict(Xn)
        print(f"{link} link, {x}: intervals use t with n - p = {f.n} - {f.p} = {f.n - f.p} df")
        for i, e in expected.iterrows():
            rows.append({"link": link, "illuminance": x, "setting": f"{e.lux} lx {e.cct} K {e.sex}", "paper": e.paper,
                         "predicted": pr["mean"][i], "95% low": pr["low"][i], "95% high": pr["high"][i],
                         "within": abs(pr["mean"][i] - e.paper) <= e.tol})
pred = pd.DataFrame(rows)
print()
print(pred.round(2).to_string(index=False))
print("\nWithin tolerance, per reading:")
print(pred.groupby(["link", "illuminance"], sort=False).within.sum().to_string())
raw = d.groupby(["lux", "cct", "sex"]).supp.agg(["count", "mean"])
print("\nVariant, raw means by sex:")
for i, e in expected.iterrows():
    m = raw.loc[(e.lux, e.cct, e.sex)]
    print(f"  {e.lux} lx {e.cct} K {e.sex}: mean {m['mean']:.2f} over {int(m['count'])} mice (paper {e.paper}),"
          f" within: {abs(m['mean'] - e.paper) <= e.tol}")

**Result: reproduced.** With the identity link and illuminance in lux the four predictions are
-10.9, 1.7, 85.6 and 98.2, all within tolerance of -9.3, 3.4, 85.7 and 98.3. The two at 50 lx
match closely and the two at 0.01 lx are 1.6 and 1.7 points lower than the paper's. No other
reading puts all four within tolerance: the log link cannot predict a negative suppression (the
male value), as the claims file noted; it puts the other three within tolerance. The raw means by
sex (variant) are -22.1, -2.5, 85.6 and 95.3, and both 0.01 lx means are outside tolerance.

## individual-variation

> Mice varied greatly in their response, individual effects ranging from up to 27.9 percentage
> points less suppression to 21.5 percentage points more suppression than average.

Criterion: from the GLM with mouse identity. Reading (a), the mouse parameters under effect coding
in the identity-link reading; reading (b), each mouse's mean predicted suppression over the 12
treatments minus the mean over all mice. Tolerances 9.55 and 13.35. Reproduced when both extremes
are within tolerance under one reading.

In [ ]:
d = a.suppression()
grid = pd.DataFrame([(lx, c) for lx in a.LUX for c in a.CCTS], columns=["lux", "cct"])
grid["loglux"] = np.log10(grid.lux)
mice = sorted(d.mouse.unique())
rows, per_mouse = [], {}
for link in ("identity", "log"):
    for x in ("lux", "loglux"):
        X, _ = a.design(d, x, "mouse")
        f = a.fit(d.supp, X, link)
        b = f.params().filter(like="mouse[")
        eff = pd.Series(np.r_[b.values, -b.sum()], index=mice)
        G = pd.concat([grid.assign(mouse=m, sex="F") for m in mice], ignore_index=True)
        Xg, _ = a.design(G, x, "mouse", means={"lux": d.lux.mean(), "loglux": d.loglux.mean()})
        G["pred"] = f.predict(Xg)["mean"].values
        mp = G.groupby("mouse").pred.mean()
        dev = mp - mp.mean()
        for reading, v in (("a, parameters", eff), ("b, mean prediction", dev)):
            if link == "log" and reading.startswith("a"):
                continue  # reading (a) is for the identity link (claims file)
            rows.append({"link": link, "illuminance": x, "reading": reading,
                         "least suppressed": v.min(), "mouse (least)": v.idxmin(),
                         "most suppressed": v.max(), "mouse (most)": v.idxmax(),
                         "both within": abs(-v.min() - 27.9) <= 9.55 and abs(v.max() - 21.5) <= 13.35})
            per_mouse[(link, x, reading)] = v
iv = pd.DataFrame(rows)
print(iv.round(2).to_string(index=False))
X, _ = a.design(d, "lux", "mouse")
f = a.fit(d.supp, X, "identity")
ci = pd.DataFrame({"effect": f.params(), "se": f.se()}).filter(like="mouse[", axis=0)
q = stats.t.ppf(0.975, f.n - f.p)
ci["low"], ci["high"] = ci.effect - q * ci.se, ci.effect + q * ci.se
print("\nIdentity link, lux, mouse parameters with 95% CI (the last mouse is minus the sum of the others):")
print(ci.round(2).to_string())
print("Last mouse,", mice[-1], ":", round(-ci.effect.sum(), 2))
print("\nRaw mean suppression per mouse minus the mean of mice (no model):")
raw = d.groupby("mouse").supp.mean()
print((raw - raw.mean()).round(2).sort_values().to_string())

**Result: reproduced.** Under reading (a), identity link and lux, the mouse parameters run from
-27.9 (6FC2, P6, less suppression) to +21.5 (6FC1, P2, more suppression), the paper's values to
the first decimal. The 95% intervals here (-38.0 to -17.7, and 7.3 to 35.7) are a little wider
than the paper's (18.3 to 37.4, and 8.1 to 34.8). Reading (b) with the identity link gives the same
values; with log10 lux the extremes are -27.2 and 18.2, also within tolerance; with the log link
reading (b) gives -14.7 and 6.8, outside it. Mouse
6FC1, P2 is the one missing all its 0.01 and 0.5 lx values, the treatments with the least
suppression, so its raw mean is high for that reason; the model's parameter accounts for the
treatments it has.

## cfos-no-cct-difference

> The number of cFos-positive cells in the SCN after 1 h of 50 lx light did not differ between
> 1950 K (123.7 +/- 17.89 s.d.) and 3000 K (113.4 +/- 8.22 s.d.) light.

Criterion (a claim of absence, on the paper's test): a two-tailed unpaired t-test with equal
variances on one value per animal, alpha 0.05; reproduced when p > 0.05; means within the paper's
s.d. of 123.7 and 113.4. The claims file's reading of the per-animal value (the mean over
observers, each observer as a variant) cannot be applied: the file holds one value per animal,
already combined.

In [ ]:
cf = a.read_csv("cFosdata.csv")
g = {c: v["Number cFos+ Cells"].values for c, v in cf.groupby("CCT")}
for c, v in g.items():
    print(f"{c} K: n {len(v)}, mean {v.mean():.2f}, s.d. {v.std(ddof=1):.2f}, s.e.m. {v.std(ddof=1) / np.sqrt(len(v)):.2f}")
t = stats.ttest_ind(g[1950], g[3000], equal_var=True)
w = stats.ttest_ind(g[1950], g[3000], equal_var=False)
print(f"\nUnpaired t-test, equal variances: t({len(cf) - 2}) = {t.statistic:.4f}, p = {t.pvalue:.4f}")
print(f"Welch variant: t = {w.statistic:.4f}, p = {w.pvalue:.4f}")
print(f"Difference of means (1950 minus 3000 K): {g[1950].mean() - g[3000].mean():.2f}")
print("Means within the paper's s.d.: 1950 K", abs(g[1950].mean() - 123.7) <= 17.89,
      "; 3000 K", abs(g[3000].mean() - 113.4) <= 8.22)
print("Shapiro-Wilk p per group:", {c: round(stats.shapiro(v).pvalue, 3) for c, v in g.items()})

In [ ]:
cf = a.read_csv("cFosdata.csv")
v = cf[cf.CCT == 3000]["Number cFos+ Cells"]
print("3000 K: paper's 's.d.' 8.22; data s.d.", round(v.std(ddof=1), 2), ", s.e.m.", round(v.std(ddof=1) / np.sqrt(len(v)), 2))

**Result: reproduced.** The t-test gives t(10) = 0.9409 and p = 0.3689, the paper's values to
four decimals, and the means are 123.74 and 113.40. The 1950 K s.d. is 17.89 as the paper gives,
but the 3000 K s.d. is 20.13; the paper's 8.22 is the s.e.m. of the 3000 K group, which is why the
claims file found the paper's means and s.d. inconsistent with its t. Two parts of the claims file
rest on reading 8.22 as an s.d.: its remark that the paper's means and s.d. give t near 1.28, and
the 8.22 tolerance on the 3000 K mean. With the s.d. of 20.13 as that tolerance the mean is still
within it, and the outcome rests on the t-test, which matches the paper, so neither changes the
outcome. The point is asked of a reader in `questions/longcore-2024.yaml`.


## Summary

| Claim | Outcome |
|---|---|
| table1-suppression-0p5-and-above | reproduced |
| table1-increase-0p01 | reproduced |
| tukey-illuminance-within-cct | not reproduced |
| cct-0p01-not-significant | reproduced |
| glm-illuminance-cct-interaction | partially reproduced |
| effect-size-illuminance-over-cct | reproduced |
| cct-effect-grows-with-intensity | not reproduced |
| lower-cct-less-suppression | reproduced |
| aicc-individual-sex | reproduced |
| sex-males-less-suppressed | reproduced |
| sex-predicted-suppression | reproduced |
| individual-variation | reproduced |
| cfos-no-cct-difference | reproduced |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning
and the differences from the paper's method.

How the analysis differs from the paper's. The models were fitted in Python with JMP's default
coding, not in JMP. The Methods state a log link, but the reported AICc differences, model
predictions (including a negative one, which a log link cannot give) and mouse effects are matched
by the identity link with illuminance in lux, and not by the log link; the claims whose criteria
allow either link are reproduced under the identity link, except glm-illuminance-cct-interaction,
which is partially reproduced under every reading (its interaction p is at least 0.0063). The abstract calls the model a "multiple linear
regression", which also points to the identity link. With lux, the log-link fit leaves its
parameters undetermined, so its Wald tests and intervals are not usable. The treatment means use 14 to 16 mice,
as the file holds, where the paper gives 16. Several of the means judged by their sign (all three
at 0.01 lx, all three at 0.5 lx and 5 lx at 1750 K) cannot be told from zero. The Tukey comparisons treat the treatments as
independent groups, as the claims file says. The cFos test uses the per-animal values as
deposited; the observers' counts are not in the dataset.

What was found about the dataset. The README gives the suppression scale as -1 to 1, but the
values are percent (-59.6 to 100). Nine of the 192 mouse-by-treatment values are missing, all for
two mice at the low illuminances, with no explanation. The dataset holds the derived suppression
values and per-animal cFos values, not the wheel counts or the per-section counts. The paper gives
the s.e.m. of the 3000 K cFos group as its s.d.

In [ ]:
print(f"Notebook ran in {(time.time() - started) / 60:.1f} min")